# Forward Dynamics - Articulated Body

Forward dynamics computation by assembling mass matrix and then conducting inversion does not scale gracefully as the number of articulated links grows. A more efficient way is to avoid explicit construction and inversion of the mass matrix while directly calculating link acceleration $\mathbf{a}$ and its projection to the joint axis $\ddot{q}$. Focusing on link $i$ in [](#rbd_aba), its Newton-Euler equation writes:

$$
\mathcal{F}_i - \mathcal{F}_{i-1} = \mathcal{M}_i\dot{\mathcal{V}}_i + \mathcal{B}_i
$$

The bias term $\mathcal{B}_i = \mathcal{V}_i \times^* \mathcal{M}_i\mathcal{V}_i$ can be obtained following the recursion in forward velocity kinematics as they are not dependent on $\ddot{q}$ and we know $\mathcal{S}_i^{\top}\mathcal{F}_i  = \tau_i$. If the effects from the successor links are null, i.e. $\mathcal{F}_{i-1}=0$, as the case of the outermost link, the link and associated joint acceleration can be easily solved. This observation implies that, for computing $\ddot{q}_i$, it might be convenient to treat all subsequent bodies as an whole _articulated body_ so the convenience of dealing with the outermost link can persist. Of course, the inertia and bias terms for this _articulated body_ might not be obvious at the moment. Nonetheless the reliance of solving $\mathcal{F}_i$ on solving $\mathcal{F}_{i-1}$ first hints that there resembles a recursive process. In fact, one can speculate that the articulated body follows a Newton-Euler equation structure like ordinary rigid bodies:

$$
\mathcal{F}_i = \mathcal{I}_i^A\dot{\mathcal{V}}_i + \mathcal{B}_i^A
$$
where $\mathcal{F}_i$ is still the force applied through joint $i$ while $\mathcal{I}_i^A$ and $\mathcal{B}_i^A$ are the inertia and bias of the articulated body. For $N$-linkage robot arm, it is obvious that $\mathcal{I}_N^A = \mathcal{M}_N$ and $\mathcal{B}_N^A = \mathcal{B}_N$. The missing step now is the recursion to compute $\mathcal{I}_i^A$ and $\mathcal{B}_i^A$ from $\mathcal{I}_{i-1}^A$ and $\mathcal{B}_{i-1}^A$. 

```{figure} ../images/rbd_aba.png
---
name: rbd_aba
---

Joint $i$ can be deemed as connecting link $i-1$ and an agglomeration of the entire sub-kinematic chain/tree. The parameters for this agglomerated link, e.g. inertias, can be recursively computed from its successor links, which can also be agglomerated links.  
```

Let's get a closer look into the link $i$ by jointly considering the two equations above:
$$
\begin{aligned}
\mathcal{F}_i & = \mathcal{M}_i\dot{\mathcal{V}}_i + \mathcal{B}_i + \mathcal{F}_{i+1}  \\
& = \mathcal{M}_i\dot{\mathcal{V}}_i + \mathcal{B}_i + \mathcal{I}_{i+1}^A\dot{\mathcal{V}}_{i+1} + \mathcal{B}_{i+1}^A 
\end{aligned}
$$
Note that $\dot{\mathcal{V}}_{i+1} = \dot{\mathcal{V}}_{i} + \ddot{q}_{i+1}\mathcal{S}_{i+1} + \dot{q}_{i+1} \mathcal{V}_{i+1} \times \mathcal{S}_{i+1}$ as in [](#eq-linkacc):

$$
\begin{aligned}
\mathcal{F}_i & = \mathcal{M}_i\dot{\mathcal{V}}_i + \mathcal{B}_i + \mathcal{I}_{i+1}^A\dot{\mathcal{V}}_{i+1}^A + \mathcal{B}_{i+1}^A \\
& = \mathcal{M}_i\dot{\mathcal{V}}_i + \mathcal{B}_i + \mathcal{I}_{i+1}^A(\dot{\mathcal{V}}_{i} + \ddot{q}_{i+1}\mathcal{S}_{i+1} + \dot{q}_{i+1} \mathcal{V}_{i+1} \times \mathcal{S}_{i+1}) + \mathcal{B}_{i+1}^A    \\
& = (\mathcal{M}_i + \mathcal{I}_{i+1}^A)\dot{\mathcal{V}}_{i} + \mathcal{B}_i + \mathcal{I}_{i+1}^A(\ddot{q}_{i+1}\mathcal{S}_{i+1} + \dot{q}_{i+1} \mathcal{V}_{i+1} \times \mathcal{S}_{i+1}) + \mathcal{B}_{i+1}^A
\end{aligned}
$$
We get to substitute $\ddot{q}_{i+1}$. To see how to do this, left multiply both sides with $\mathcal{S}^{\top}_{i+1}$ to [](#eq-newtoneuler_ab) with the index of $i+1$, and use a more compact representation with $\mathcal{C}_i = \dot{q}_{i} \mathcal{V}_{i} \times \mathcal{S}_{i}$:

$$
\begin{aligned}
\tau_{i+1} & = \mathcal{S}^{\top}_{i+1}\mathcal{F}_{i+1} = \mathcal{S}^{\top}_{i+1}\mathcal{I}_{i+1}^A\dot{\mathcal{V}}_{i+1} + \mathcal{S}^{\top}_{i+1}\mathcal{B}_{i+1}^A \\
\quad & = \mathcal{S}^{\top}_{i+1}\mathcal{I}_{i+1}^A(\dot{\mathcal{V}}_{i} + \ddot{q}_{i+1}\mathcal{S}_{i+1} + \mathcal{C}_{i+1}) + \mathcal{S}^{\top}_{i+1}\mathcal{B}_{i+1}^A  \\
\Rightarrow \\
\ddot{q}_{i+1} & = (\mathcal{S}^{\top}_{i+1}\mathcal{I}_{i+1}^A\mathcal{S}_{i+1})^{-1}[\tau_{i+1} - \mathcal{S}^{\top}_{i+1}\mathcal{I}_{i+1}^A\dot{\mathcal{V}}_{i} - \mathcal{S}^{\top}_{i+1}(\mathcal{B}_{i+1}^A + \mathcal{I}_{i+1}^A\mathcal{C}_{i+1})] \\
\Rightarrow \\
\mathcal{F}_i & = (\mathcal{M}_i + \mathcal{I}_{i+1}^A)\dot{\mathcal{V}}_{i} + \mathcal{B}_i + \mathcal{I}_{i+1}^A\mathcal{S}_{i+1}\frac{\tau_{i+1} - \mathcal{S}^{\top}_{i+1}\mathcal{I}_{i+1}^A\dot{\mathcal{V}}_{i} - \mathcal{S}^{\top}_{i+1}(\mathcal{B}_{i+1}^A + \mathcal{I}_{i+1}^A\mathcal{C}_{i+1})}{\mathcal{S}^{\top}_{i+1}\mathcal{I}_{i+1}^A\mathcal{S}_{i+1}} + \mathcal{I}_{i+1}^A\mathcal{C}_{i+1} + \mathcal{B}_{i+1}^A   \\
& = \underbrace{(\mathcal{M}_i + \mathcal{I}_{i+1}^A - \frac{\mathcal{I}_{i+1}^A\mathcal{S}_{i+1}\mathcal{S}^{\top}_{i+1}\mathcal{I}_{i+1}^A}{\mathcal{S}^{\top}_{i+1}\mathcal{I}_{i+1}^A\mathcal{S}_{i+1}})}_{\mathcal{I}_i^{A}}\dot{\mathcal{V}}_{i} 
+ \underbrace{\mathcal{B}_i + \mathcal{B}_{i+1}^A +  \mathcal{I}_{i+1}^A\mathcal{C}_{i+1} + \frac{\mathcal{I}_{i+1}^A\mathcal{S}_{i+1}(\tau_{i+1} - \mathcal{S}^{\top}_{i+1}(\mathcal{B}_{i+1}^A + \mathcal{I}_{i+1}^A\mathcal{C}_{i+1}))}{\mathcal{S}^{\top}_{i+1}\mathcal{I}_{i+1}^A\mathcal{S}_{i+1}}}_{\mathcal{B}_i^A}
\end{aligned}
$$

The equation seems horrendous but resembles yet another Newton-Euler equation. And more importantly, the articulated body inertia and bias terms indeed reflect the recursion structure:

$$
\begin{aligned}
\mathcal{I}_i^A &   \leftarrow \mathcal{M}_i + \mathcal{I}_{i+1}^A - \frac{\mathcal{I}_{i+1}^A\mathcal{S}_{i+1}\mathcal{S}^{\top}_{i+1}\mathcal{I}_{i+1}^A}{\mathcal{S}^{\top}_{i+1}\mathcal{I}_{i+1}^A\mathcal{S}_{i+1}}   \\
\mathcal{B}_i^A &   \leftarrow  \mathcal{B}_i + \mathcal{B}_{i+1}^A +  \mathcal{I}_{i+1}^A\mathcal{C}_{i+1} + \frac{\mathcal{I}_{i+1}^A\mathcal{S}_{i+1}(\tau_{i+1} - \mathcal{S}^{\top}_{i+1}(\mathcal{B}_{i+1}^A + \mathcal{I}_{i+1}^A\mathcal{C}_{i+1}))}{\mathcal{S}^{\top}_{i+1}\mathcal{I}_{i+1}^A\mathcal{S}_{i+1}}
\end{aligned}
$$

The overall algorithm for computing $\ddot{\mathbf{q}}$ given $\mathbf{q}$, $\dot{\mathbf{q}}$ and $\mathbf{\tau}$ is hence consisted of following steps:

*   Compute all acceleration independent terms like transforms between adjacent links ${}^{i-1}\mathbf{T}_{i}$ and link velocities $\mathcal{V}_i$. This can be done via a forward recursion as in forward velocity kinematics.
*   Compute all articulated body inertia and bias terms using [](#eq-aba_recursion). This boils down to a backward recursion from $N$. 
*   Compute $\dot{\mathcal{V}_i}$ through [](#eq-newtoneuler_ab) and $\ddot{q}_i$ via [](#eq-aba_qdd). Just another loop over all the links.

The complexity is finalized as $O(N)$ in contrast to $O(N^3)$ for inversing generalized mass matrix. Inversing $\mathcal{S}^{\top}_{i+1}\mathcal{I}_{i+1}^A\mathcal{S}_{i+1}$ is with a constant dimension and simply a reciprocal for 1-DOF joints. Note that to avoid further mess, the equation notations assume reference frames are already aligned for computations like $\mathcal{M}_i + \mathcal{I}_{i+1}^A$ to be valid. This three-loop algorithm is called __Articulated Body Algorithm (ABA)__ featured in [@featherstone2007]. The complete and side-by-side comparison between equations and implementable pseudo-code can be found in pp. 137. 

:::{note}
The idea of ignoring internal interactions between links but focusing on effects external to the whole articulated body is a system perspective, commonly used in physics for force analysis. Actually, the idea of using unit accelerate rate to reconstruct the generalized mass matrix undergos a similar process of aggregating link masses in the sub-chain/tree. See the part about __Composite-Rigid-Body Algorithm (CRBA)__ in  [@featherstone2007]. ABA provides a superior complexity while CRBA gives $\mathbf{M}$ as a byproduct which can find useful in many cases. It is also possible to accelerate linear solving by exploiting sparsity so the $O(N^3)$ algorithm still works fine for moderate link numbers.   
:::